In [1]:
import pandas as pd

In [2]:
df_test=pd.read_csv('ru_train_s.csv')

In [3]:
df_test

,Text2,Text1,Score
0,"Если это когда-нибудь случится, просто выключи...","Это случается, просто выключи трубку.",1.00
1,Актёр Бен Газзара умирает в 81 году.,Актёр Газзара мертв в 81-м.,1.00
2,Бледная собака бежит по тропинке.,Легкая коричневая собака с радостью бежит по т...,1.00
3,"Ребята могут быть вялыми, я должен знать.",Ребята могут быть странными.,1.00
4,Человек на фермерском рынке.,Человек на фермерском рынке.,1.00
...,...,...,...
1095,Жизнь не всегда такая.,Это не англичанинское предложение.,0.06
1096,Мальчик с черным мусором на спине.,Самый замечательный азиатский кухонный двор!,0.03
1097,Одна собака кусается в лицо другой собаке на т...,Человек формирует дерево с латой в своем рабоч...,0.03
1098,"Его мама - англичанин, его отец - испанец.","Это то же самое, что и в английской пицце Хат.",0.03


In [4]:
from sentence_transformers import SentenceTransformer, util
import torch

/home/kirill/anaconda3/envs/testtf/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-01-28 10:54:54.932202: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [5]:
model = embedding_model = SentenceTransformer(model_name_or_path="ai-forever/ru-en-RoSBERTa", 
                                      device="cuda") 

def compute_cosine_similarity(row):
    embedding_1 = model.encode(row['Text1'], convert_to_tensor=True)
    embedding_2 = model.encode(row['Text2'], convert_to_tensor=True)
    similarity = util.pytorch_cos_sim(embedding_1, embedding_2)
    return similarity.item()

# Create a new column 'CosineSimilarity' and apply the function
df_test['CosineSimilarity_ru-en-RoSBERTa'] = df_test.apply(compute_cosine_similarity, axis=1)

Some weights of RobertaModel were not initialized from the model checkpoint at ai-forever/ru-en-RoSBERTa and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
model = embedding_model = SentenceTransformer(model_name_or_path="ai-forever/sbert_large_mt_nlu_ru", 
                                      device="cuda") 

def compute_cosine_similarity(row):
    embedding_1 = model.encode(row['Text1'], convert_to_tensor=True)
    embedding_2 = model.encode(row['Text2'], convert_to_tensor=True)
    similarity = util.pytorch_cos_sim(embedding_1, embedding_2)
    return similarity.item()

# Create a new column 'CosineSimilarity' and apply the function
df_test['CosineSimilarity_sbert_large_mt_nlu_ru'] = df_test.apply(compute_cosine_similarity, axis=1)

In [7]:
model = embedding_model = SentenceTransformer(model_name_or_path="sentence-transformers/all-MiniLM-L6-v2", 
                                      device="cuda") 

def compute_cosine_similarity(row):
    embedding_1 = model.encode(row['Text1'], convert_to_tensor=True)
    embedding_2 = model.encode(row['Text2'], convert_to_tensor=True)
    similarity = util.pytorch_cos_sim(embedding_1, embedding_2)
    return similarity.item()

# Create a new column 'CosineSimilarity' and apply the function
df_test['CosineSimilarity_all-MiniLM-L6-v2'] = df_test.apply(compute_cosine_similarity, axis=1)

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 87cbc518-9d05-48ce-8447-3c69f7b85c5b)')' thrown while requesting GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/README.md
Retrying in 1s [Retry 1/5].


In [8]:
df_test

,Text2,Text1,Score,CosineSimilarity_ru-en-RoSBERTa,CosineSimilarity_sbert_large_mt_nlu_ru,CosineSimilarity_all-MiniLM-L6-v2
0,"Если это когда-нибудь случится, просто выключи...","Это случается, просто выключи трубку.",1.00,0.854233,0.763137,0.812231
1,Актёр Бен Газзара умирает в 81 году.,Актёр Газзара мертв в 81-м.,1.00,0.885999,0.855109,0.851446
2,Бледная собака бежит по тропинке.,Легкая коричневая собака с радостью бежит по т...,1.00,0.788628,0.804778,0.859042
3,"Ребята могут быть вялыми, я должен знать.",Ребята могут быть странными.,1.00,0.783596,0.669603,0.862554
4,Человек на фермерском рынке.,Человек на фермерском рынке.,1.00,1.000000,1.000000,1.000000
...,...,...,...,...,...,...
1095,Жизнь не всегда такая.,Это не англичанинское предложение.,0.06,0.437034,0.201833,0.631701
1096,Мальчик с черным мусором на спине.,Самый замечательный азиатский кухонный двор!,0.03,0.424698,0.215273,0.672724
1097,Одна собака кусается в лицо другой собаке на т...,Человек формирует дерево с латой в своем рабоч...,0.03,0.404055,0.233545,0.697892
1098,"Его мама - англичанин, его отец - испанец.","Это то же самое, что и в английской пицце Хат.",0.03,0.536669,0.305388,0.618062


In [ ]:
from scipy.stats import spearmanr

In [11]:
correlation, _ = spearmanr(df_test['Score'], df_test['CosineSimilarity_ru-en-RoSBERTa'])

print(f"Spearman Correlation ru-en-RoSBERTa: {correlation}")

Spearman Correlation ru-en-RoSBERTa: 0.7647094357794535


In [12]:
correlation, _ = spearmanr(df_test['Score'], df_test['CosineSimilarity_sbert_large_mt_nlu_ru'])

print(f"Spearman Correlation sbert_large_mt_nlu_ru: {correlation}")

Spearman Correlation sbert_large_mt_nlu_ru: 0.697585698400043


In [13]:
correlation, _ = spearmanr(df_test['Score'], df_test['CosineSimilarity_all-MiniLM-L6-v2'])

print(f"Spearman Correlation all-MiniLM-L6-v2: {correlation}")

Spearman Correlation all-MiniLM-L6-v2: 0.527541839823828


In [49]:
from scipy import stats

In [50]:
correlation, _ = stats.pearsonr(df_test['Score'], df_test['CosineSimilarity_ru-en-RoSBERTa'])

print(f"Pearson Correlation ru-en-RoSBERTa: {correlation}")

Pearson Correlation ru-en-RoSBERTa: 0.7701285865868752


In [51]:
correlation, _ = stats.pearsonr(df_test['Score'], df_test['CosineSimilarity_sbert_large_mt_nlu_ru'])

print(f"Pearson Correlation sbert_large_mt_nlu_ru: {correlation}")

Pearson Correlation sbert_large_mt_nlu_ru: 0.7027918470427509


In [52]:
correlation, _ = stats.pearsonr(df_test['Score'], df_test['CosineSimilarity_all-MiniLM-L6-v2'])

print(f"Pearson Correlation all-MiniLM-L6-v2: {correlation}")

Pearson Correlation all-MiniLM-L6-v2: 0.5180673160046977


In [45]:
l1=[]
l2=[]
l3=[]
for i in range(len(df_test)):
    l1.append(df_test.iloc[i]['Score']-df_test.iloc[i]['CosineSimilarity_ru-en-RoSBERTa'])
    l2.append(df_test.iloc[i]['Score']-df_test.iloc[i]['CosineSimilarity_sbert_large_mt_nlu_ru'])
    l3.append(df_test.iloc[i]['Score']-df_test.iloc[i]['CosineSimilarity_all-MiniLM-L6-v2'])



In [31]:
import numpy as np

In [46]:
for i in range(len(l1)):
    l1[i]=l1[i]**2
    l2[i]=l2[i]**2
    l3[i]=l3[i]**2

In [47]:
print(f"MSE ru-en-RoSBERTa: {sum(l1)/len(l1)}")
print(f"MSE sbert_large_mt_nlu_ru: {sum(l2)/len(l2)}")
print(f"MSE all-MiniLM-L6-v2: {sum(l3)/len(l3)}")

MSE ru-en-RoSBERTa: 0.044452667577451546
MSE sbert_large_mt_nlu_ru: 0.03189910025556855
MSE all-MiniLM-L6-v2: 0.07222048380610988


In [48]:
print(f"RMSE ru-en-RoSBERTa: {(sum(l1)/len(l1))**(1/2)}")
print(f"RMSE sbert_large_mt_nlu_ru: {(sum(l2)/len(l2))**(1/2)}")
print(f"RMSE all-MiniLM-L6-v2: {(sum(l3)/len(l3))**(1/2)}")

RMSE ru-en-RoSBERTa: 0.21083801264822136
RMSE sbert_large_mt_nlu_ru: 0.17860319217631176
RMSE all-MiniLM-L6-v2: 0.26873869056410515
